# Subscription Renewal Prediction

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `renewed`

## 2 · Project Overview

We predict whether a subscriber will **renew** their subscription based on tenure, usage, charges, support tickets, and plan type.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a subscriber's usage patterns, tenure, monthly charge, support ticket history, and plan tier, predict whether they will renew their subscription.

## 5 · Why This Project Matters

- **Revenue retention** is cheaper than new customer acquisition.
- Predicting non-renewals enables proactive outreach.
- Understanding renewal drivers helps optimize pricing and support.
- Directly impacts subscription business KPIs.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 6,000 |
| **Features** | tenure_months, monthly_charge, usage_hours, support_tickets, plan |
| **Target** | renewed (0 = not renewed, 1 = renewed) |
| **Class balance** | ~50/50 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "renewed"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: renewed
Save dir: E:\Github\Machine-Learning-Projects\Classification\Subscription Renewal Prediction


## 11 · Dataset Download or Loading

Synthetic dataset: 6,000 subscribers with usage, tenure, and renewal outcome.

In [4]:
np.random.seed(SEED)
n = 6000
tenure_months = np.random.randint(1, 60, n)
monthly_charge = np.round(np.random.uniform(10, 120, n), 2)
usage_hours = np.round(np.random.exponential(20, n), 1)
support_tickets = np.random.poisson(1.5, n)
plan = np.random.choice(["Basic", "Standard", "Premium"], n, p=[0.4, 0.35, 0.25])
plan_num = np.where(plan == "Basic", 0, np.where(plan == "Standard", 1, 2))

score = (0.02 * tenure_months + 0.01 * usage_hours - 0.005 * monthly_charge
         - 0.15 * support_tickets + 0.3 * plan_num + np.random.normal(0, 0.8, n))
renewed = (score > np.median(score)).astype(int)

df = pd.DataFrame({"tenure_months": tenure_months, "monthly_charge": monthly_charge,
                    "usage_hours": usage_hours, "support_tickets": support_tickets,
                    "plan": plan, "renewed": renewed})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['renewed'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (6000, 6)
Class balance:
renewed
1    0.5
0    0.5
Name: proportion, dtype: float64


,tenure_months,monthly_charge,usage_hours,support_tickets,plan,renewed
0,39,95.41,8.0,3,Basic,1
1,52,77.25,3.9,0,Premium,1
2,29,104.30,29.0,3,Basic,0
3,15,84.06,30.9,2,Basic,0
4,43,54.89,16.7,2,Standard,1


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (6000, 6)

Missing values:
tenure_months      0
monthly_charge     0
usage_hours        0
support_tickets    0
plan               0
renewed            0
dtype: int64

Duplicate rows: 0

Dtypes:
tenure_months        int32
monthly_charge     float64
usage_hours        float64
support_tickets      int32
plan                object
renewed              int64
dtype: object

Target 'renewed' confirmed. Value counts:
renewed
1    3000
0    3000
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = df.select_dtypes(include="number").columns.drop(TARGET).tolist()
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, col in enumerate(num_cols[:4]):
    ax = axes[i // 2][i % 2]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
plt.suptitle("Feature Distributions by Renewal Status", fontsize=14)
plt.tight_layout()
plt.show()

corr = df[num_cols + [TARGET]].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `renewed`.

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color=["salmon", "steelblue"], edgecolor="black")
ax.set_title("Renewal Distribution")
ax.set_xticklabels(["Not Renewed (0)", "Renewed (1)"], rotation=0)
plt.tight_layout()
plt.show()
print(f"Class balance: {dict(df[TARGET].value_counts())}")

Class balance: {1: np.int64(3000), 0: np.int64(3000)}


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (4800, 5), Test: (1200, 5)
Train class distribution:
renewed
0    0.5
1    0.5
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None in synthetic data.
- **Categorical**: `plan` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree-based models.
- **Class balance**: ~50/50 by design.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["charge_per_month_tenure"] = X_train["monthly_charge"] / (X_train["tenure_months"] + 1)
X_test["charge_per_month_tenure"] = X_test["monthly_charge"] / (X_test["tenure_months"] + 1)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (6): ['tenure_months', 'monthly_charge', 'usage_hours', 'support_tickets', 'plan', 'charge_per_month_tenure']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.6758
F1       : 0.6644

              precision    recall  f1-score   support

           0       0.66      0.71      0.69       600
           1       0.69      0.64      0.66       600

    accuracy                           0.68      1200
   macro avg       0.68      0.68      0.68      1200
weighted avg       0.68      0.68      0.68      1200



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
AdaBoostClassifier             0.694167           0.694167  0.765742  0.693906   0.694830  0.694167    0.174549
LGBMClassifier                 0.687500           0.687500  0.756224  0.686793   0.689208  0.687500    3.236266
GaussianNB                     0.685833           0.685833  0.743558  0.682160   0.694840  0.685833    0.015886
CatBoostClassifier             0.684167           0.684167  0.761656  0.683348   0.686090  0.684167    2.728981
SVC                            0.680833           0.680833  0.754500  0.679712   0.683401  0.680833    1.351781
NearestCentroid                0.680000           0.680000  0.739300  0.679978   0.680050  0.680000    0.017633
QuadraticDiscriminantAnalysis  0.678333           0.678333  0.743267  

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='binary'):.4f}")

FLAML best model: lgbm
Accuracy: 0.6817
F1: 0.6701


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.6661  (1.3s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[70]	valid_0's binary_logloss: 0.580948
LightGBM F1: 0.6625  (0.8s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="binary"), 4),
        "Precision": round(precision_score(y_test, yp, average="binary", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="binary", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
FLAML                  0.6817  0.6701     0.6953  0.6467
CatBoost               0.6833  0.6661     0.7045  0.6317
Logistic Regression    0.6758  0.6644     0.6887  0.6417
LightGBM               0.6833  0.6625     0.7091  0.6217

Top 3 models: ['FLAML', 'CatBoost', 'Logistic Regression']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='binary'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='binary', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='binary', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  FLAML:
    Accuracy : 0.6817
    F1       : 0.6701
    Precision: 0.6953
    Recall   : 0.6467

  CatBoost:
    Accuracy : 0.6833
    F1       : 0.6661
    Precision: 0.7045
    Recall   : 0.6317

  Logistic Regression:
    Accuracy : 0.6758
    F1       : 0.6644
    Precision: 0.6887
    Recall   : 0.6417


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: FLAML

Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.72      0.69       600
           1       0.70      0.65      0.67       600

    accuracy                           0.68      1200
   macro avg       0.68      0.68      0.68      1200
weighted avg       0.68      0.68      0.68      1200


Total errors: 382 / 1200 (31.8%)

Sample misclassifications:
      tenure_months  monthly_charge  usage_hours  support_tickets  plan  charge_per_month_tenure  actual  predicted  correct
4644             37           42.15         14.5                1   0.0                 1.109211       1          0    False
5810             22           21.76         22.9                0   0.0                 0.946087       0          1    False
3186             52           41.70         10.9                2   0.0                 0.786792       0          1    False
2720              1           69.01         17.0                2   1

## 25 · Interpretation and Business Insight

**Key findings:**
- **Tenure and usage** are the strongest renewal predictors.
- **Support tickets** negatively correlate with renewal.
- **Premium plans** have higher renewal rates.

**Business takeaway:** Focus retention efforts on low-tenure, high-ticket subscribers.

## 26 · Limitations

1. Synthetic data with simplified patterns.
2. No seasonality or contract terms.
3. Missing behavioral signals (login frequency, feature usage).
4. No competitor pricing context.
5. Binary outcome ignores partial renewals or downgrades.

## 27 · How to Improve This Project

1. Add real subscription data with behavioral signals.
2. Include contract length and auto-renewal flags.
3. Add time-series features (usage trend over months).
4. Try survival analysis for time-to-churn.
5. A/B test retention interventions.

## 28 · Production Considerations

- Retrain monthly as subscriber behavior shifts.
- Monitor for concept drift in renewal rates.
- Output renewal probability, not just binary prediction.
- Integrate with CRM for automated outreach triggers.

## 29 · Common Mistakes

1. Ignoring class imbalance when present.
2. Using accuracy alone for evaluation.
3. Not segmenting by plan type.
4. Leaking future data (e.g., using renewal month features).
5. Over-engineering on synthetic data.

## 30 · Mini Challenge / Exercises

1. Remove `plan` and retrain — how much does F1 drop?
2. Create tenure buckets (0-12, 13-36, 37+) and compare renewal rates.
3. Try SMOTE on an imbalanced version (keep only 20% renewals).
4. Add interaction: `usage_hours * tenure_months`.
5. Plot feature importances from the best tree model.

## 31 · Final Summary / Key Takeaways

1. **Tenure and usage** are the strongest renewal predictors.
2. **Support tickets** signal disengagement.
3. **Tree-based models** handle mixed features well.
4. **LazyPredict + FLAML** provide rapid benchmarking.
5. **Proactive retention** based on model scores can reduce churn.
6. **Always compare against baseline** — improvements must be meaningful.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="binary")), 4),
        "precision": round(float(precision_score(y_test, yp, average="binary", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="binary", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Subscription Renewal Prediction\metrics.json
{
  "CatBoost": {
    "accuracy": 0.6833,
    "f1": 0.6661,
    "precision": 0.7045,
    "recall": 0.6317
  },
  "LightGBM": {
    "accuracy": 0.6833,
    "f1": 0.6625,
    "precision": 0.7091,
    "recall": 0.6217
  },
  "Logistic Regression": {
    "accuracy": 0.6758,
    "f1": 0.6644,
    "precision": 0.6887,
    "recall": 0.6417
  },
  "FLAML": {
    "accuracy": 0.6817,
    "f1": 0.6701,
    "precision": 0.6953,
    "recall": 0.6467
  }
}
